In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql.functions import broadcast


In [2]:
spark = SparkSession.builder.appName("NYC-traffic-collision").getOrCreate()

# Reading data

In [3]:
#df_nyc_traffic = spark.read\
#.option("delimiter",",")\
#.option("header", True)\
#.option("quote", '"')\
#.option("escape", '"')\
#.option("multiline", True)\
#.csv("dataset/Motor_Vehicle_Collisions_-_Crashes.csv")\
#.withColumn("CRASH_DATE",f.to_date(f.col("CRASH DATE"),"MM/dd/yyyy"))\
#.filter(f.col("CRASH_DATE").isNotNull())\
#.filter( (f.col("CRASH_DATE") >= '2024-01-01') & (f.col("CRASH_DATE") <= '2024-12-31')).show()
#.coalesce(1)\
#.write\
#.option("delimiter", ",") \
#.option("header", True) \
#.mode("overwrite") \
#.csv("dataset/Motor_Vehicle_Collisions_-_Crashes_2024")


In [4]:
df_nyc_traffic = spark.read\
.option("delimiter",",")\
.option("header", True)\
.option("quote", '"')\
.option("escape", '"')\
.option("multiline", True)\
.csv("dataset/Motor_Vehicle_Collisions_-_Crashes_2024.csv")

In [5]:
df_nyc_traffic.show(2, truncate=False)

+----------+----------+-------+--------+--------+---------+---------------------+-----------------------+-----------------+---------------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+------------------------------+-----------------------------+-----------------------------+-----------------------------+-----------------------------+------------+-------------------+-------------------+-------------------+-------------------+-------------------+----------+
|CRASH DATE|CRASH TIME|BOROUGH|ZIP CODE|LATITUDE|LONGITUDE|LOCATION             |ON STREET NAME         |CROSS STREET NAME|OFF STREET NAME|NUMBER OF PERSONS INJURED|NUMBER OF PERSONS KILLED|NUMBER OF PEDESTRIANS INJURED|NUMBER OF PEDESTRIANS KILLED|NUMBER OF CYCLIST INJURED|NUMBER OF CYCLIST KILLED|NUMBER OF MOTORIST INJURED|NUMBER OF MOTORIST KILLED|CONTRIBUTING FACTO

In [6]:
import requests
import json

def fetch_data(url: str):
    response = requests.get(url)
    response.raise_for_status()
    return  response.json()


year = 2024
country_code = 'US'
pub_holidays = fetch_data(f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}")


In [7]:
pub_holidays

[{'date': '2024-01-01',
  'localName': "New Year's Day",
  'name': "New Year's Day",
  'countryCode': 'US',
  'fixed': False,
  'global': True,
  'counties': None,
  'launchYear': None,
  'types': ['Public']},
 {'date': '2024-01-15',
  'localName': 'Martin Luther King, Jr. Day',
  'name': 'Martin Luther King, Jr. Day',
  'countryCode': 'US',
  'fixed': False,
  'global': True,
  'counties': None,
  'launchYear': None,
  'types': ['Public']},
 {'date': '2024-02-12',
  'localName': "Lincoln's Birthday",
  'name': "Lincoln's Birthday",
  'countryCode': 'US',
  'fixed': False,
  'global': False,
  'counties': ['US-CA',
   'US-CT',
   'US-IL',
   'US-IN',
   'US-KY',
   'US-MI',
   'US-NY',
   'US-MO',
   'US-OH'],
  'launchYear': None,
  'types': ['Observance']},
 {'date': '2024-02-19',
  'localName': "Washington's Birthday",
  'name': 'Presidents Day',
  'countryCode': 'US',
  'fixed': False,
  'global': True,
  'counties': None,
  'launchYear': None,
  'types': ['Public']},
 {'date': '20

In [8]:
response_schema = {
  "type": "array",
  "items": {
    "required": [
      "date",
      "localName",
      "name",
      "countryCode",
      "types"
    ],
    "type": "object",
    "properties": {
      "date": {
        "type": "string",
        "description": "The date of the holiday.",
        "format": "date"
      },
      "localName": {
        "type": "string",
        "description": "Local name of the holiday."
      },
      "name": {
        "type": "string",
        "description": "English name of the holiday."
      },
      "countryCode": {
        "type": "string",
        "description": "ISO 3166-1 alpha-2 country code."
      },
      "fixed": {
        "type": "boolean",
        "description": "Indicates if this holiday occurs on the same date every year."
      },
      "global": {
        "type": "boolean",
        "description": "Indicates if this holiday applies to the entire country."
      },
      "counties": {
        "type": [
          "null",
          "array"
        ],
        "items": {
          "type": "string"
        },
        "description": "ISO-3166-2 codes of the subdivisions where this holiday applies"
      },
      "launchYear": {
        "type": [
          "null",
          "integer"
        ],
        "description": "The year the holiday was first observed.",
        "format": "int32"
      },
      "types": {
        "type": "array",
        "items": {
          "enum": [
            "Public",
            "Bank",
            "School",
            "Authorities",
            "Optional",
            "Observance"
          ],
          "type": "string",
          "format": "string"
        },
        "description": "List of holiday types this holiday is classified under."
      }
    },
    "description": "Represents a public holiday."
  }
}


from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, ArrayType

schema = StructType([
    StructField("date", StringType(), True),
    StructField("localName", StringType(), True),
    StructField("name", StringType(), True),
    StructField("countryCode", StringType(), True),
    StructField("fixed", BooleanType(), True),
    StructField("global", BooleanType(), True),
    StructField("counties", ArrayType(StringType()), True),
    StructField("launchYear", IntegerType(), True),
    StructField("types", ArrayType(StringType()), True),
])

In [9]:
df_holidays = spark.createDataFrame(pub_holidays,schema)

In [10]:
df_holidays.show(truncate=False)

+----------+------------------------------------+------------------------------------+-----------+-----+------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+---------------------+
|date      |localName                           |name                                |countryCode|fixed|global|counties                                                                                                                                                                                                                               |launchYear|types                |
+----------+------------------------------------+------------------------------------+-----------+-----+------+---------------------------------------------------------------------------------------------------------------------------------------

# Exploring data and fixing data

In [11]:
df_nyc_traffic.show(10, truncate=False)

+----------+----------+-------+--------+---------+---------+----------------------+-----------------------+----------------------+---------------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+------------------------------+-----------------------------+-----------------------------+-----------------------------+-----------------------------+------------+-----------------------------------+-----------------------------------+-------------------+-------------------+-------------------+----------+
|CRASH DATE|CRASH TIME|BOROUGH|ZIP CODE|LATITUDE |LONGITUDE|LOCATION              |ON STREET NAME         |CROSS STREET NAME     |OFF STREET NAME|NUMBER OF PERSONS INJURED|NUMBER OF PERSONS KILLED|NUMBER OF PEDESTRIANS INJURED|NUMBER OF PEDESTRIANS KILLED|NUMBER OF CYCLIST INJURED|NUMBER OF CYCLIST KILLED|NUMBER OF MOTORIST INJURE

In [12]:
df_nyc_traffic.schema

StructType([StructField('CRASH DATE', StringType(), True), StructField('CRASH TIME', StringType(), True), StructField('BOROUGH', StringType(), True), StructField('ZIP CODE', StringType(), True), StructField('LATITUDE', StringType(), True), StructField('LONGITUDE', StringType(), True), StructField('LOCATION', StringType(), True), StructField('ON STREET NAME', StringType(), True), StructField('CROSS STREET NAME', StringType(), True), StructField('OFF STREET NAME', StringType(), True), StructField('NUMBER OF PERSONS INJURED', StringType(), True), StructField('NUMBER OF PERSONS KILLED', StringType(), True), StructField('NUMBER OF PEDESTRIANS INJURED', StringType(), True), StructField('NUMBER OF PEDESTRIANS KILLED', StringType(), True), StructField('NUMBER OF CYCLIST INJURED', StringType(), True), StructField('NUMBER OF CYCLIST KILLED', StringType(), True), StructField('NUMBER OF MOTORIST INJURED', StringType(), True), StructField('NUMBER OF MOTORIST KILLED', StringType(), True), StructFiel

In [13]:
df_nyc_traffic.count()

91314

### cleaning crash date and crash time

In [14]:
df_nyc_traffic.select("CRASH DATE").distinct().show()

+----------+
|CRASH DATE|
+----------+
|02/04/2024|
|01/11/2024|
|12/28/2024|
|12/14/2024|
|01/29/2024|
|12/30/2024|
|07/09/2024|
|02/18/2024|
|11/04/2024|
|01/28/2024|
|03/28/2024|
|01/27/2024|
|05/24/2024|
|10/18/2024|
|06/18/2024|
|06/01/2024|
|05/28/2024|
|07/30/2024|
|09/30/2024|
|01/12/2024|
+----------+
only showing top 20 rows



In [15]:
df_nyc_traffic.filter(f.col("CRASH DATE").isNull()).count()

0

In [16]:
df_nyc_traffic.withColumn("CRASH_DATE",f.to_date(f.col("CRASH DATE"),"MM/dd/yyyy"))\
.select("CRASH DATE","CRASH_DATE")\
.filter(f.col("CRASH_DATE").isNull()).count()

0

In [17]:
df_nyc_traffic.withColumn("CRASH_DATE",f.to_date(f.col("CRASH DATE"),"MM/dd/yyyy"))\
.select("CRASH DATE","CRASH_DATE")\
.filter(f.col("CRASH_DATE").isNull()).show()

+----------+----------+
|CRASH DATE|CRASH_DATE|
+----------+----------+
+----------+----------+



In [18]:
df_nyc_traffic.withColumn("CRASH_DATE",f.to_date(f.col("CRASH DATE"),"MM/dd/yyyy"))\
.filter(f.col("CRASH_DATE").isNull() ).show()

+----------+----------+-------+--------+--------+---------+--------+--------------+-----------------+---------------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+-----------------------------+-----------------------------+-----------------------------+-----------------------------+-----------------------------+------------+-------------------+-------------------+-------------------+-------------------+-------------------+----------+
|CRASH DATE|CRASH TIME|BOROUGH|ZIP CODE|LATITUDE|LONGITUDE|LOCATION|ON STREET NAME|CROSS STREET NAME|OFF STREET NAME|NUMBER OF PERSONS INJURED|NUMBER OF PERSONS KILLED|NUMBER OF PEDESTRIANS INJURED|NUMBER OF PEDESTRIANS KILLED|NUMBER OF CYCLIST INJURED|NUMBER OF CYCLIST KILLED|NUMBER OF MOTORIST INJURED|NUMBER OF MOTORIST KILLED|CONTRIBUTING FACTOR VEHICLE 1|CONTRIBUTING FACTOR VEHICLE 2|CON

In [19]:
df_nyc_traffic\
.withColumn("CRASH_HOUR", f.hour(f.to_timestamp(f.col("CRASH TIME"),"H:m")))\
.select("CRASH_HOUR","CRASH TIME")


DataFrame[CRASH_HOUR: int, CRASH TIME: string]

In [20]:
df_nyc_traffic\
.withColumn("CRASH_DATE", f.to_date(f.col("CRASH DATE"), "MM/dd/yyyy"))\
.withColumn("CRASH_HOUR", f.hour(f.to_timestamp(f.col("CRASH TIME"),"H:m")))\
.select("CRASH_DATE","CRASH_HOUR")\
.schema

StructType([StructField('CRASH_DATE', DateType(), True), StructField('CRASH_HOUR', IntegerType(), True)])

## exploring borough and zipcode , location, latitudine, longitudine 

In [21]:
df_nyc_traffic.filter(f.col("BOROUGH").isNull()).count()

26085

In [22]:
df_nyc_traffic.groupBy("BOROUGH").count().show()

+-------------+-----+
|      BOROUGH|count|
+-------------+-----+
|         NULL|26085|
|       QUEENS|17811|
|     BROOKLYN|22786|
|        BRONX|10032|
|    MANHATTAN|11904|
|STATEN ISLAND| 2696|
+-------------+-----+



In [23]:
df_nyc_traffic.filter(f.col("ZIP CODE").isNull()).count()

26099

In [24]:
df_nyc_traffic.groupBy("ZIP CODE").count().show()

+--------+-----+
|ZIP CODE|count|
+--------+-----+
|   10309|  232|
|   11236| 1067|
|   11205|  399|
|   11106|  339|
|   11251|    1|
|   10110|    1|
|   11218|  536|
|   10452|  389|
|   11428|  155|
|   10169|    1|
|   11237|  458|
|   10177|    1|
|   11379|  211|
|   11364|  176|
|   11109|    5|
|   11249|  281|
|   10012|  256|
|   11001|   24|
|   10039|  156|
|   11385|  830|
+--------+-----+
only showing top 20 rows



In [25]:
df_nyc_traffic.groupBy("LOCATION","LATITUDE").count().orderBy(f.col("count").desc()).show()

+--------------------+---------+-----+
|            LOCATION| LATITUDE|count|
+--------------------+---------+-----+
|                NULL|     NULL| 7096|
|          (0.0, 0.0)|        0|  528|
|(40.815754, -73.8...|40.815754|   67|
|(40.861862, -73.9...|40.861862|   49|
|(40.668507, -73.9...|40.668507|   42|
|(40.71976, -73.94...| 40.71976|   36|
|(40.64254, -73.87...| 40.64254|   35|
|(40.675735, -73.8...|40.675735|   33|
|(40.804585, -73.9...|40.804585|   32|
|(40.813095, -73.8...|40.813095|   31|
|(40.608223, -74.1...|40.608223|   31|
|(40.71122, -73.72...| 40.71122|   30|
|(40.651974, -73.8...|40.651974|   29|
|(40.70313, -73.81...| 40.70313|   29|
|(40.704494, -73.8...|40.704494|   29|
|(40.764267, -73.7...|40.764267|   28|
|(40.696033, -73.9...|40.696033|   28|
|(40.651863, -73.8...|40.651863|   27|
|(40.60567, -74.03...| 40.60567|   27|
|(40.669476, -73.9...|40.669476|   27|
+--------------------+---------+-----+
only showing top 20 rows



## cast field number of ---

In [26]:
df_nyc_traffic\
.select("NUMBER OF PERSONS INJURED","NUMBER OF PERSONS KILLED","NUMBER OF PEDESTRIANS INJURED","NUMBER OF PEDESTRIANS KILLED","NUMBER OF CYCLIST INJURED","NUMBER OF CYCLIST KILLED","NUMBER OF MOTORIST INJURED","NUMBER OF MOTORIST KILLED")\
.withColumn("NUMBER_OF_PERSONS_INJURED",f.col("NUMBER OF PERSONS INJURED").cast("integer"))\
.withColumn("NUMBER_OF_PERSONS_KILLED",f.col("NUMBER OF PERSONS KILLED").cast("integer"))\
.withColumn("NUMBER_OF_PEDESTRIANS_INJURED",f.col("NUMBER OF PEDESTRIANS INJURED").cast("integer"))\
.withColumn("NUMBER_OF_PEDESTRIANS_KILLED",f.col("NUMBER OF PEDESTRIANS KILLED").cast("integer"))\
.withColumn("NUMBER_OF_CYCLIST_INJURED",f.col("NUMBER OF CYCLIST INJURED").cast("integer"))\
.withColumn("NUMBER_OF_CYCLIST_KILLED",f.col("NUMBER OF CYCLIST KILLED").cast("integer"))\
.withColumn("NUMBER_OF_MOTORIST_INJURED",f.col("NUMBER OF MOTORIST INJURED").cast("integer"))\
.withColumn("NUMBER_OF_MOTORIST_KILLED",f.col("NUMBER OF MOTORIST KILLED").cast("integer"))\
.fillna(0)\
.select("NUMBER_OF_PERSONS_INJURED","NUMBER_OF_PERSONS_KILLED","NUMBER_OF_PEDESTRIANS_INJURED","NUMBER_OF_PEDESTRIANS_KILLED","NUMBER_OF_CYCLIST_INJURED","NUMBER_OF_CYCLIST_KILLED","NUMBER_OF_MOTORIST_INJURED","NUMBER_OF_MOTORIST_KILLED")\
.show()

+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+
|NUMBER_OF_PERSONS_INJURED|NUMBER_OF_PERSONS_KILLED|NUMBER_OF_PEDESTRIANS_INJURED|NUMBER_OF_PEDESTRIANS_KILLED|NUMBER_OF_CYCLIST_INJURED|NUMBER_OF_CYCLIST_KILLED|NUMBER_OF_MOTORIST_INJURED|NUMBER_OF_MOTORIST_KILLED|
+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+
|                        0|                       0|                            0|                           0|                        0|                       0|                         0|                        0|
|                        2|                       0|                            0|                           0|                        1

In [27]:
df_nyc_traffic.select("CONTRIBUTING FACTOR VEHICLE 1","CONTRIBUTING FACTOR VEHICLE 2","CONTRIBUTING FACTOR VEHICLE 3","CONTRIBUTING FACTOR VEHICLE 4","CONTRIBUTING FACTOR VEHICLE 5")\
.show(truncate=False)

+------------------------------+-----------------------------+-----------------------------+-----------------------------+-----------------------------+
|CONTRIBUTING FACTOR VEHICLE 1 |CONTRIBUTING FACTOR VEHICLE 2|CONTRIBUTING FACTOR VEHICLE 3|CONTRIBUTING FACTOR VEHICLE 4|CONTRIBUTING FACTOR VEHICLE 5|
+------------------------------+-----------------------------+-----------------------------+-----------------------------+-----------------------------+
|Passing or Lane Usage Improper|Failure to Yield Right-of-Way|NULL                         |NULL                         |NULL                         |
|Unspecified                   |Unspecified                  |NULL                         |NULL                         |NULL                         |
|Driver Inattention/Distraction|Unspecified                  |NULL                         |NULL                         |NULL                         |
|Driver Inattention/Distraction|NULL                         |NULL                

## holiday 

In [28]:
df_holidays.show()

+----------+--------------------+--------------------+-----------+-----+------+--------------------+----------+--------------------+
|      date|           localName|                name|countryCode|fixed|global|            counties|launchYear|               types|
+----------+--------------------+--------------------+-----------+-----+------+--------------------+----------+--------------------+
|2024-01-01|      New Year's Day|      New Year's Day|         US|false|  true|                NULL|      NULL|            [Public]|
|2024-01-15|Martin Luther Kin...|Martin Luther Kin...|         US|false|  true|                NULL|      NULL|            [Public]|
|2024-02-12|  Lincoln's Birthday|  Lincoln's Birthday|         US|false| false|[US-CA, US-CT, US...|      NULL|        [Observance]|
|2024-02-19|Washington's Birt...|      Presidents Day|         US|false|  true|                NULL|      NULL|            [Public]|
|2024-03-29|         Good Friday|         Good Friday|         US|fal

In [29]:
df_holidays.schema

StructType([StructField('date', StringType(), True), StructField('localName', StringType(), True), StructField('name', StringType(), True), StructField('countryCode', StringType(), True), StructField('fixed', BooleanType(), True), StructField('global', BooleanType(), True), StructField('counties', ArrayType(StringType(), True), True), StructField('launchYear', IntegerType(), True), StructField('types', ArrayType(StringType(), True), True)])

In [30]:
df_holidays.filter(f.col("date").isNull()).count()

0

In [31]:
df_holidays.select(f.col("types")).distinct().show(truncate=False)

+---------------------+
|types                |
+---------------------+
|[Public]             |
|[Observance]         |
|[Optional]           |
|[School, Authorities]|
+---------------------+



# data cleaning and layer1

In [32]:
relevant_cols = ["CRASH_DATE","BOROUGH","ZIP CODE","NUMBER_OF_PERSONS_INJURED","NUMBER_OF_PERSONS_KILLED","NUMBER_OF_PEDESTRIANS_INJURED","NUMBER_OF_PEDESTRIANS_KILLED",
                 "NUMBER_OF_CYCLIST_INJURED","NUMBER_OF_CYCLIST_KILLED","NUMBER_OF_MOTORIST_INJURED","NUMBER_OF_MOTORIST_KILLED","TIME_OF_DAY"]
'''
part_of_day
early-morning: 8 - 10
late-morning: 11 - 13 
early-afternoon: 14 - 16
late-afternoon: 17 - 19
evening: 20 - 22
night 22 - 7
'''
df_nyc_traffic_l1 = df_nyc_traffic\
.withColumn("NUMBER_OF_PERSONS_INJURED",f.col("NUMBER OF PERSONS INJURED").cast("integer"))\
.withColumn("NUMBER_OF_PERSONS_KILLED",f.col("NUMBER OF PERSONS KILLED").cast("integer"))\
.withColumn("NUMBER_OF_PEDESTRIANS_INJURED",f.col("NUMBER OF PEDESTRIANS INJURED").cast("integer"))\
.withColumn("NUMBER_OF_PEDESTRIANS_KILLED",f.col("NUMBER OF PEDESTRIANS KILLED").cast("integer"))\
.withColumn("NUMBER_OF_CYCLIST_INJURED",f.col("NUMBER OF CYCLIST INJURED").cast("integer"))\
.withColumn("NUMBER_OF_CYCLIST_KILLED",f.col("NUMBER OF CYCLIST KILLED").cast("integer"))\
.withColumn("NUMBER_OF_MOTORIST_INJURED",f.col("NUMBER OF MOTORIST INJURED").cast("integer"))\
.withColumn("NUMBER_OF_MOTORIST_KILLED",f.col("NUMBER OF MOTORIST KILLED").cast("integer"))\
.fillna(0)\
.withColumn("CRASH_HOUR", f.hour(f.to_timestamp(f.col("CRASH TIME"),"H:m")))\
.withColumn("TIME_OF_DAY", f.when( (f.col("CRASH_HOUR") >= 8) & (f.col("CRASH_HOUR") < 11),"early-morning")
                        .when( (f.col("CRASH_HOUR") >= 11) & (f.col("CRASH_HOUR") < 14),"late-morning")\
                        .when( (f.col("CRASH_HOUR") >= 14) & (f.col("CRASH_HOUR") < 17),"early-afternoon")\
                        .when( (f.col("CRASH_HOUR") >= 17) & (f.col("CRASH_HOUR") < 20),"late-afternoon")\
                        .when( (f.col("CRASH_HOUR") >= 20) & (f.col("CRASH_HOUR") < 23),"evening")\
                        .when( (f.col("CRASH_HOUR") >= 23) & (f.col("CRASH_HOUR") < 8),"night")\
                        .otherwise(None)\
          )\
.withColumn("CRASH_DATE",f.to_date(f.col("CRASH DATE"),"MM/dd/yyyy"))\
.filter(f.col("CRASH_DATE").isNotNull())\
.select(relevant_cols)


In [33]:
relevant_holidays_cols = ["date","isPublic","isLocal"] #"global","counties","launchYear","types"]

df_holidays_l1 = df_holidays\
.withColumn("isPublic", f.array_contains(f.col("types"),"Public"))\
.withColumn("isLocal", (f.col("counties").isNotNull() & f.array_contains(f.col("counties"),"US-NY")))\
.select(relevant_holidays_cols)\

df_holidays_l1.show(truncate=False)

+----------+--------+-------+
|date      |isPublic|isLocal|
+----------+--------+-------+
|2024-01-01|true    |false  |
|2024-01-15|true    |false  |
|2024-02-12|false   |true   |
|2024-02-19|true    |false  |
|2024-03-29|true    |false  |
|2024-03-29|false   |false  |
|2024-05-08|false   |false  |
|2024-05-27|true    |false  |
|2024-06-19|true    |false  |
|2024-07-04|true    |false  |
|2024-09-02|true    |false  |
|2024-10-14|true    |true   |
|2024-10-14|true    |false  |
|2024-11-11|true    |false  |
|2024-11-28|true    |false  |
|2024-12-25|true    |false  |
+----------+--------+-------+



## joining data & new kpi

In [34]:
df_nyc_traffic_l2 = df_nyc_traffic_l1.join(f.broadcast(df_holidays_l1), df_nyc_traffic_l1.CRASH_DATE==df_holidays_l1.date, how="left")\
.withColumn("isHoliday",f.col("date").isNotNull())\
.drop("date")\
.groupBy("CRASH_DATE","BOROUGH","ZIP CODE","TIME_OF_DAY","isPublic","isLocal","isHoliday")\
.agg(f.sum("NUMBER_OF_PERSONS_INJURED").alias("NUMBER_OF_PERSONS_INJURED"),
     f.sum("NUMBER_OF_PERSONS_KILLED").alias("NUMBER_OF_PERSONS_KILLED"),
     f.sum("NUMBER_OF_PEDESTRIANS_INJURED").alias("NUMBER_OF_PEDESTRIANS_INJURED"),
     f.sum("NUMBER_OF_PEDESTRIANS_KILLED").alias("NUMBER_OF_PEDESTRIANS_KILLED"),
     f.sum("NUMBER_OF_CYCLIST_INJURED").alias("NUMBER_OF_CYCLIST_INJURED"),
     f.sum("NUMBER_OF_CYCLIST_KILLED").alias("NUMBER_OF_CYCLIST_KILLED"),
     f.sum("NUMBER_OF_MOTORIST_INJURED").alias("NUMBER_OF_MOTORIST_INJURED"),
     f.sum("NUMBER_OF_MOTORIST_KILLED").alias("NUMBER_OF_MOTORIST_KILLED")
    )

In [35]:
df_nyc_traffic_l2.show(2)

+----------+-------+--------+--------------+--------+-------+---------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+
|CRASH_DATE|BOROUGH|ZIP CODE|   TIME_OF_DAY|isPublic|isLocal|isHoliday|NUMBER_OF_PERSONS_INJURED|NUMBER_OF_PERSONS_KILLED|NUMBER_OF_PEDESTRIANS_INJURED|NUMBER_OF_PEDESTRIANS_KILLED|NUMBER_OF_CYCLIST_INJURED|NUMBER_OF_CYCLIST_KILLED|NUMBER_OF_MOTORIST_INJURED|NUMBER_OF_MOTORIST_KILLED|
+----------+-------+--------+--------------+--------+-------+---------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+
|2024-05-08|  BRONX|   10460|       evening|   false|  false|     true|                        3|                       0|                    

In [36]:
df_nyc_traffic_l2.filter(f.col("TIME_OF_DAY").isNotNull()).show()

+----------+-------------+--------+---------------+--------+-------+---------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+
|CRASH_DATE|      BOROUGH|ZIP CODE|    TIME_OF_DAY|isPublic|isLocal|isHoliday|NUMBER_OF_PERSONS_INJURED|NUMBER_OF_PERSONS_KILLED|NUMBER_OF_PEDESTRIANS_INJURED|NUMBER_OF_PEDESTRIANS_KILLED|NUMBER_OF_CYCLIST_INJURED|NUMBER_OF_CYCLIST_KILLED|NUMBER_OF_MOTORIST_INJURED|NUMBER_OF_MOTORIST_KILLED|
+----------+-------------+--------+---------------+--------+-------+---------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+
|2024-05-08|        BRONX|   10460|        evening|   false|  false|     true|                        3|                 

In [37]:
df_nyc_traffic_l2.printSchema()

root
 |-- CRASH_DATE: date (nullable = true)
 |-- BOROUGH: string (nullable = true)
 |-- ZIP CODE: string (nullable = true)
 |-- TIME_OF_DAY: string (nullable = true)
 |-- isPublic: boolean (nullable = true)
 |-- isLocal: boolean (nullable = true)
 |-- isHoliday: boolean (nullable = false)
 |-- NUMBER_OF_PERSONS_INJURED: long (nullable = true)
 |-- NUMBER_OF_PERSONS_KILLED: long (nullable = true)
 |-- NUMBER_OF_PEDESTRIANS_INJURED: long (nullable = true)
 |-- NUMBER_OF_PEDESTRIANS_KILLED: long (nullable = true)
 |-- NUMBER_OF_CYCLIST_INJURED: long (nullable = true)
 |-- NUMBER_OF_CYCLIST_KILLED: long (nullable = true)
 |-- NUMBER_OF_MOTORIST_INJURED: long (nullable = true)
 |-- NUMBER_OF_MOTORIST_KILLED: long (nullable = true)



In [40]:
spark.read.parquet("output").show()

+----------+-------------+--------+---------------+--------+-------+---------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+
|CRASH_DATE|      BOROUGH|ZIP CODE|    TIME_OF_DAY|isPublic|isLocal|isHoliday|NUMBER_OF_PERSONS_INJURED|NUMBER_OF_PERSONS_KILLED|NUMBER_OF_PEDESTRIANS_INJURED|NUMBER_OF_PEDESTRIANS_KILLED|NUMBER_OF_CYCLIST_INJURED|NUMBER_OF_CYCLIST_KILLED|NUMBER_OF_MOTORIST_INJURED|NUMBER_OF_MOTORIST_KILLED|
+----------+-------------+--------+---------------+--------+-------+---------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+
|2024-05-08|        BRONX|   10460|        evening|   false|  false|     true|                        3|                 